In [ ]:
import setup_utils
import experiment_lib
import os

# --- CONFIGURATION ---
REPO_URL = "https://github.com/fabianandresgrob/gaussian-splatting.git" # Update if needed
REPO_BRANCH = "main" # Update if needed
DATA_ROOT = "/content/drive/MyDrive/Scannet++" # Update with your Drive path
OUTPUT_ROOT = "/content/drive/MyDrive/3DGS_Results"

# 1. Setup Environment
repo_path = setup_utils.init_colab_env(REPO_URL, REPO_BRANCH)

In [ ]:
# 2. Define Experiments
SCENES = ["0c5385e84b"] # Add more scene IDs here
STRATEGIES = ["random", "fixed_prob", "clustering"] # Add your strategies
SEEDS = [0, 1, 2, 3, 4]

In [ ]:
# 3. Run Loop
for scene in SCENES:
    scene_path = os.path.join(DATA_ROOT, scene, "dslr") # Scannet++ structure
    
    for strategy in STRATEGIES:
        exp_name = f"{scene}_{strategy}"
        
        for seed in SEEDS:
            experiment_lib.run_training(
                repo_path=repo_path,
                data_path=scene_path,
                output_dir=OUTPUT_ROOT,
                strategy=strategy,
                seed=seed,
                exp_name=exp_name
            )

Results Directory Structure
===========================
```
/content/drive/MyDrive/3DGS_Results/       <-- Your OUTPUT_ROOT_DRIVE
│
├── 0c5385e84b_random/                     <-- Experiment Folder ({scene}_{strategy})
│   ├── seed_0/                            <-- Run Folder (Per Seed)
│   │   ├── point_cloud/
│   │   │   └── iteration_30000/
│   │   │       └── point_cloud.ply        <-- Final trained Gaussian model
│   │   ├── eval/
│   │   │   ├── 00000.png                  <-- Rendered test image 0
│   │   │   ├── 00001.png
│   │   │   └── ...
│   │   ├── combine/
│   │   │   ├── 00000.png                  <-- Side-by-side: Render vs. GT
│   │   │   └── ...
│   │   ├── metrics_history.json           <-- Training curve data (Loss/PSNR over time)
│   │   ├── final_results.json             <-- Final averaged metrics for this seed
│   │   ├── console_log.txt                <-- Full training log output
│   │   ├── cameras.json                   <-- Camera parameters used
│   │   ├── cfg_args                       <-- Arguments used for this run
│   │   ├── chkpnt30000.pth                <-- PyTorch checkpoint
│   │   └── events.out.tfevents...         <-- Tensorboard logs
│   │
│   ├── seed_1/
│   │   └── ... (Same structure)
│   └── ...
│
├── 0c5385e84b_fixed_prob/                 <-- Next Experiment
│   └── ...
│
└── experiment_summary.csv                 <-- Final aggregated report of all runs
```

In [ ]:
# 4. Analysis & Report
import pandas as pd
results = []
for scene in SCENES:
    for strategy in STRATEGIES:
        rep = experiment_lib.aggregate_results(OUTPUT_ROOT, scene, strategy)
        if rep: results.append(rep)

df = pd.DataFrame(results)
print(df)
df.to_csv(os.path.join(OUTPUT_ROOT, "final_experiment_report.csv"))